# SECTION 0

## SECTION 0.1 – Core imports & basic settings

In [1]:
import os
import sys
import warnings

import numpy as np
import pandas as pd
from tqdm import tqdm

pd.set_option("display.max_columns", None)

warnings.simplefilter(action="ignore", category=pd.errors.PerformanceWarning)

## SECTION 0.2 – Paths and global configuration

In [2]:
DATA_FOLDER = "./data-master"

SPADL_H5       = os.path.join(DATA_FOLDER, "spadl-statsbomb.h5")
FEATURES_H5    = os.path.join(DATA_FOLDER, "features.h5")
LABELS_H5      = os.path.join(DATA_FOLDER, "labels.h5")
PREDICTIONS_H5 = os.path.join(DATA_FOLDER, "predictions.h5")

RNG_SEED = 42

TRAIN_RATIO = 0.70
TEST_RATIO  = 0.20
VAL_RATIO   = 0.10

## SECTION 0.3 – Make sure the new data folder exists

In [3]:
if not os.path.exists(DATA_FOLDER):
    os.mkdir(DATA_FOLDER)
    print(f"Created NEW data folder: {DATA_FOLDER}")
else:
    print(f"Using existing data folder: {DATA_FOLDER}")


Created NEW data folder: ./data-master


## SECTION 0.4 – Football (socceraction) + VAEP + model imports

In [6]:
import socceraction.spadl as spadl
from socceraction.data.statsbomb import StatsBombLoader

from socceraction.vaep import features as fs
from socceraction.vaep import labels as lab
from socceraction.vaep import formula as vaepformula

from xgboost import XGBClassifier
from catboost import CatBoostClassifier

from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score

print("Imports successful.")

Imports successful.


# Section 1

## SECTION 1.1 – Set up the StatsBomb data loader

In [7]:
loader = StatsBombLoader()

competitions = loader.competitions()

print("Available competitions:")
display(competitions[["competition_id", "season_id", "competition_name", "season_name"]])


C:\Users\Arya1\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsbombpy\api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


Available competitions:


,competition_id,season_id,competition_name,season_name
0,9,281,1. Bundesliga,2023/2024
1,9,27,1. Bundesliga,2015/2016
2,1267,107,African Cup of Nations,2023
3,16,4,Champions League,2018/2019
4,16,1,Champions League,2017/2018
...,...,...,...,...
70,35,75,UEFA Europa League,1988/1989
71,53,315,UEFA Women's Euro,2025
72,53,106,UEFA Women's Euro,2022
73,72,107,Women's World Cup,2023


## SECTION 1.2 – Select one competition + one season

In [8]:
selected_comp = 43
selected_season = 3

games = loader.games(selected_comp, selected_season)

print("Number of games:", len(games))
games.head()


C:\Users\Arya1\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsbombpy\api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


Number of games: 64


,game_id,season_id,competition_id,competition_stage,game_day,game_date,home_team_id,away_team_id,home_score,away_score,venue,referee
0,7585,3,43,Round of 16,4,2018-07-03 20:00:00,769,768,1,1,Otkritie Bank Arena,Mark Geiger
1,7570,3,43,Group Stage,3,2018-06-28 20:00:00,768,782,0,1,Stadion Kaliningrad,Damir Skomina
2,7586,3,43,Round of 16,4,2018-07-03 16:00:00,790,773,1,0,Saint-Petersburg Stadium,Damir Skomina
3,7557,3,43,Group Stage,3,2018-06-25 20:00:00,797,780,1,1,Mordovia Arena,Enrique Cáceres
4,7542,3,43,Group Stage,2,2018-06-20 14:00:00,780,788,1,0,Stadion Luzhniki,Mark Geiger


## SECTION 1.3 – Convert all games to SPADL and store in spadl-statsbomb.h5

In [12]:
with pd.HDFStore(SPADL_H5) as spadlstore:
    
    spadlstore["competitions"] = competitions
    spadlstore["games"] = games

    for game_id in tqdm(games.game_id, desc="Converting games to SPADL"):
        
        events = loader.events(game_id)

        home_team_id = int(games.loc[games.game_id == game_id, "home_team_id"].iloc[0])

        actions = spadl.statsbomb.convert_to_actions(events, home_team_id)

        key = f"actions/game_{int(game_id)}"
        spadlstore[key] = actions

print(f"Saved SPADL data to: {SPADL_H5}")

Converting games to SPADL:   0%|                                                                | 0/64 [00:00<?, ?it/s]C:\Users\Arya1\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsbombpy\api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
C:\Users\Arya1\AppData\Local\Programs\Python\Python311\Lib\site-packages\socceraction\data\statsbomb\loader.py:337: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  eventsdf["under_pressure"] = eventsdf["under_pressure"].fillna(False).astype(bool)
C:\Users\Arya1\AppData\Local\Programs\Python\Python311\Lib\site-packages\socceraction\data\statsbomb\loader.py:338: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a fut

Saved SPADL data to: ./data-master\spadl-statsbomb.h5


In [13]:
with pd.HDFStore(SPADL_H5) as store:
    first_game_id = int(games.game_id.iloc[0])
    key = f"actions/game_{first_game_id}"
    sample_actions = store[key]

sample_actions.head()

,game_id,original_event_id,period_id,time_seconds,team_id,player_id,start_x,start_y,end_x,end_y,type_id,result_id,bodypart_id,action_id
0,7585,d4883f20-ce68-4f84-b26a-a049a13cb6be,1,0.24,769,3445.0,52.0625,34.425,43.3125,33.575,0,1,4,0
1,7585,b948f032-4c54-4782-a71a-ffeed8908d00,1,0.48,769,5692.0,43.3125,33.575,44.1875,34.425,21,1,0,1
2,7585,9bdb71f9-c87b-4a66-96f0-def5312ca921,1,2.12,769,5692.0,44.1875,34.425,40.6875,22.525,0,1,4,2
3,7585,2ffa2904-8b47-4817-af26-aa9ac8d2881a,1,3.44,769,5685.0,40.6875,22.525,42.4375,21.675,21,1,0,3
4,7585,6cb0d85d-bd14-42e3-9c2d-7f99ce437796,1,4.20,769,5685.0,42.4375,21.675,56.4375,1.275,0,1,5,4


# Section 2

## SECTION 2.1 – Load games from the SPADL file

In [14]:
print("SPADL file:", SPADL_H5)

games = pd.read_hdf(SPADL_H5, "games")
print("Number of games in SPADL file:", len(games))

games.head()

SPADL file: ./data-master\spadl-statsbomb.h5
Number of games in SPADL file: 64


,game_id,season_id,competition_id,competition_stage,game_day,game_date,home_team_id,away_team_id,home_score,away_score,venue,referee
0,7585,3,43,Round of 16,4,2018-07-03 20:00:00,769,768,1,1,Otkritie Bank Arena,Mark Geiger
1,7570,3,43,Group Stage,3,2018-06-28 20:00:00,768,782,0,1,Stadion Kaliningrad,Damir Skomina
2,7586,3,43,Round of 16,4,2018-07-03 16:00:00,790,773,1,0,Saint-Petersburg Stadium,Damir Skomina
3,7557,3,43,Group Stage,3,2018-06-25 20:00:00,797,780,1,1,Mordovia Arena,Enrique Cáceres
4,7542,3,43,Group Stage,2,2018-06-20 14:00:00,780,788,1,0,Stadion Luzhniki,Mark Geiger


## SECTION 2.2 – Compute FEATURES for every action in every game

In [15]:
xfns = [
    fs.actiontype,
    fs.actiontype_onehot,
    fs.bodypart,
    fs.bodypart_onehot,
    fs.result,
    fs.result_onehot,
    fs.goalscore,
    fs.startlocation,
    fs.endlocation,
    fs.movement,
    fs.space_delta,
    fs.startpolar,
    fs.endpolar,
    fs.team,
    fs.time,
    fs.time_delta,
]

print("Features will be stored in:", FEATURES_H5)

with pd.HDFStore(SPADL_H5) as spadlstore, pd.HDFStore(FEATURES_H5) as featurestore:
    for game in tqdm(list(games.itertuples()), desc=f"Generating and storing features in {FEATURES_H5}"):

        actions = spadlstore[f"actions/game_{game.game_id}"]

        actions_named = spadl.add_names(actions)

        gamestates = fs.gamestates(actions_named, 3)

        gamestates = fs.play_left_to_right(gamestates, game.home_team_id)

        X = pd.concat([fn(gamestates) for fn in xfns], axis=1)

        featurestore.put(f"game_{game.game_id}", X, format="table")

print("Finished computing features.")


Features will be stored in: ./data-master\features.h5


Generating and storing features in ./data-master\features.h5:   0%|                             | 0/64 [00:00<?, ?it/s]C:\Users\Arya1\AppData\Local\Programs\Python\Python311\Lib\site-packages\socceraction\vaep\features.py:93: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  prev_actions = actions.groupby(["game_id", "period_id"], sort=False, as_index=False).apply(
C:\Users\Arya1\AppData\Local\Programs\Python\Python311\Lib\site-packages\socceraction\vaep\features.py:93: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude 

Finished computing features.


## SECTION 2.3 – Compute LABELS for every action in every game

In [16]:
yfns = [
    lab.scores,
    lab.concedes,
    lab.goal_from_shot,
]

print("Labels will be stored in:", LABELS_H5)

with pd.HDFStore(SPADL_H5) as spadlstore, pd.HDFStore(LABELS_H5) as labelstore:
    for game in tqdm(list(games.itertuples()), desc=f"Computing and storing labels in {LABELS_H5}"):

        actions = spadlstore[f"actions/game_{game.game_id}"]

        actions_named = spadl.add_names(actions)

        Y = pd.concat([fn(actions_named) for fn in yfns], axis=1)

        labelstore.put(f"game_{game.game_id}", Y, format="table")

print("Finished computing labels.")


Labels will be stored in: ./data-master\labels.h5


Computing and storing labels in ./data-master\labels.h5: 100%|█████████████████████████| 64/64 [00:02<00:00, 22.63it/s]

Finished computing labels.


## SECTION 2.3 – Compute LABELS for every action in every game

In [ ]:
yfns = [
    lab.scores,
    lab.concedes,
    lab.goal_from_shot,
]

print("Labels will be stored in:", LABELS_H5)

with pd.HDFStore(SPADL_H5) as spadlstore, pd.HDFStore(LABELS_H5) as labelstore:
    for game in tqdm(list(games.itertuples()), desc=f"Computing and storing labels in {LABELS_H5}"):
        actions = spadlstore[f"actions/game_{game.game_id}"]

        actions_named = spadl.add_names(actions)

        Y = pd.concat([fn(actions_named) for fn in yfns], axis=1)

        labelstore.put(f"game_{game.game_id}", Y, format="table")

print("Finished computing labels.")


## SECTION 2.4 – Quick checks on features and labels for one game

In [17]:
first_game_id = int(games.game_id.iloc[0])
key = f"game_{first_game_id}"

with pd.HDFStore(FEATURES_H5) as fstore:
    X_sample = fstore[key].head()

with pd.HDFStore(LABELS_H5) as lstore:
    Y_sample = lstore[key].head()

print("Sample FEATURES (X):")
display(X_sample)

print("Sample LABELS (Y):")
display(Y_sample)


Sample FEATURES (X):


,actiontype_a0,actiontype_a1,actiontype_a2,actiontype_pass_a0,actiontype_cross_a0,actiontype_throw_in_a0,actiontype_freekick_crossed_a0,actiontype_freekick_short_a0,actiontype_corner_crossed_a0,actiontype_corner_short_a0,actiontype_take_on_a0,actiontype_foul_a0,actiontype_tackle_a0,actiontype_interception_a0,actiontype_shot_a0,actiontype_shot_penalty_a0,actiontype_shot_freekick_a0,actiontype_keeper_save_a0,actiontype_keeper_claim_a0,actiontype_keeper_punch_a0,actiontype_keeper_pick_up_a0,actiontype_clearance_a0,actiontype_bad_touch_a0,actiontype_non_action_a0,actiontype_dribble_a0,actiontype_goalkick_a0,actiontype_pass_a1,actiontype_cross_a1,actiontype_throw_in_a1,actiontype_freekick_crossed_a1,actiontype_freekick_short_a1,actiontype_corner_crossed_a1,actiontype_corner_short_a1,actiontype_take_on_a1,actiontype_foul_a1,actiontype_tackle_a1,actiontype_interception_a1,actiontype_shot_a1,actiontype_shot_penalty_a1,actiontype_shot_freekick_a1,actiontype_keeper_save_a1,actiontype_keeper_claim_a1,actiontype_keeper_punch_a1,actiontype_keeper_pick_up_a1,actiontype_clearance_a1,actiontype_bad_touch_a1,actiontype_non_action_a1,actiontype_dribble_a1,actiontype_goalkick_a1,actiontype_pass_a2,actiontype_cross_a2,actiontype_throw_in_a2,actiontype_freekick_crossed_a2,actiontype_freekick_short_a2,actiontype_corner_crossed_a2,actiontype_corner_short_a2,actiontype_take_on_a2,actiontype_foul_a2,actiontype_tackle_a2,actiontype_interception_a2,actiontype_shot_a2,actiontype_shot_penalty_a2,actiontype_shot_freekick_a2,actiontype_keeper_save_a2,actiontype_keeper_claim_a2,actiontype_keeper_punch_a2,actiontype_keeper_pick_up_a2,actiontype_clearance_a2,actiontype_bad_touch_a2,actiontype_non_action_a2,actiontype_dribble_a2,actiontype_goalkick_a2,bodypart_a0,bodypart_a1,bodypart_a2,bodypart_foot_a0,bodypart_head_a0,bodypart_other_a0,bodypart_head/other_a0,bodypart_foot_a1,bodypart_head_a1,bodypart_other_a1,bodypart_head/other_a1,bodypart_foot_a2,bodypart_head_a2,bodypart_other_a2,bodypart_head/other_a2,result_a0,result_a1,result_a2,result_fail_a0,result_success_a0,result_offside_a0,result_owngoal_a0,result_yellow_card_a0,result_red_card_a0,result_fail_a1,result_success_a1,result_offside_a1,result_owngoal_a1,result_yellow_card_a1,result_red_card_a1,result_fail_a2,result_success_a2,result_offside_a2,result_owngoal_a2,result_yellow_card_a2,result_red_card_a2,goalscore_team,goalscore_opponent,goalscore_diff,start_x_a0,start_y_a0,start_x_a1,start_y_a1,start_x_a2,start_y_a2,end_x_a0,end_y_a0,end_x_a1,end_y_a1,end_x_a2,end_y_a2,dx_a0,dy_a0,movement_a0,dx_a1,dy_a1,movement_a1,dx_a2,dy_a2,movement_a2,dx_a01,dy_a01,mov_a01,dx_a02,dy_a02,mov_a02,start_dist_to_goal_a0,start_angle_to_goal_a0,start_dist_to_goal_a1,start_angle_to_goal_a1,start_dist_to_goal_a2,start_angle_to_goal_a2,end_dist_to_goal_a0,end_angle_to_goal_a0,end_dist_to_goal_a1,end_angle_to_goal_a1,end_dist_to_goal_a2,end_angle_to_goal_a2,team_1,team_2,period_id_a0,time_seconds_a0,time_seconds_overall_a0,period_id_a1,time_seconds_a1,time_seconds_overall_a1,period_id_a2,time_seconds_a2,time_seconds_overall_a2,time_delta_1,time_delta_2
0,pass,pass,pass,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,foot,foot,foot,True,False,False,False,True,False,False,False,True,False,False,False,success,success,success,False,True,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,0,0,0,52.0625,34.425,52.0625,34.425,52.0625,34.425,43.3125,33.575,43.3125,33.575,43.3125,33.575,-8.750,-0.85,8.791189,-8.750,-0.85,8.791189,-8.750,-0.85,8.791189,-8.75,-0.85,8.791189,-8.750,-0.85,8.791189,52.939206,0.008028,52.939206,0.008028,52.939

Sample LABELS (Y):


,scores,concedes,goal_from_shot
0,False,False,False
1,False,False,False
2,False,False,False
3,False,False,False
4,False,False,False


## Section 3

## SECTION 3.1 – Build the full X (features) and Y (labels) across all games

In [18]:
def build_full_XY(features_h5, labels_h5, games):

    X_list = []
    Y_list = []
    game_id_list = []

    with pd.HDFStore(features_h5) as fstore, pd.HDFStore(labels_h5) as lstore:
        for game in tqdm(list(games.itertuples()), desc="Loading features & labels from HDF5"):
            game_id = int(game.game_id)

            Xg = fstore[f"game_{game_id}"]
            Yg = lstore[f"game_{game_id}"][["scores", "concedes"]] 

            assert len(Xg) == len(Yg), f"Row mismatch in game {game_id}"

            X_list.append(Xg)
            Y_list.append(Yg)

            game_id_list.append(np.full(len(Xg), game_id, dtype=int))

    X_all = pd.concat(X_list, ignore_index=True)
    Y_all = pd.concat(Y_list, ignore_index=True)
    game_ids = pd.Series(np.concatenate(game_id_list), name="game_id")

    return X_all, Y_all, game_ids


X_all, Y_all, game_ids = build_full_XY(FEATURES_H5, LABELS_H5, games)

print("Full X shape:", X_all.shape)
print("Full Y shape:", Y_all.shape)
print("Game_ids length:", game_ids.shape)


Loading features & labels from HDF5: 100%|█████████████████████████████████████████████| 64/64 [00:04<00:00, 12.96it/s]

Full X shape: (128484, 163)
Full Y shape: (128484, 2)
Game_ids length: (128484,)


## SECTION 3.2 – Split by game into Train / Test / Validation

In [19]:
rng = np.random.RandomState(RNG_SEED)

unique_games = games.game_id.unique()
n_games = len(unique_games)

rng.shuffle(unique_games)

n_train = int(TRAIN_RATIO * n_games)
n_test  = int(TEST_RATIO  * n_games)
n_val   = n_games - n_train - n_test   # whatever is left

train_games = unique_games[:n_train]
test_games  = unique_games[n_train:n_train + n_test]
val_games   = unique_games[n_train + n_test:]

print(f"Total games:   {n_games}")
print(f"Train games:   {len(train_games)}")
print(f"Test games:    {len(test_games)}")
print(f"Val games:     {len(val_games)}")


Total games:   64
Train games:   44
Test games:    12
Val games:     8


## SECTION 3.3 – Build Train / Test / Validation sets for X and Y

In [20]:
train_mask = game_ids.isin(train_games)
test_mask  = game_ids.isin(test_games)
val_mask   = game_ids.isin(val_games)

X_train, Y_train = X_all[train_mask], Y_all[train_mask]
X_test,  Y_test  = X_all[test_mask],  Y_all[test_mask]
X_val,   Y_val   = X_all[val_mask],   Y_all[val_mask]

print("Train X shape:", X_train.shape, "| Train Y shape:", Y_train.shape)
print("Test  X shape:", X_test.shape,  "| Test  Y shape:", Y_test.shape)
print("Val   X shape:", X_val.shape,   "| Val   Y shape:", Y_val.shape)


Train X shape: (88516, 163) | Train Y shape: (88516, 2)
Test  X shape: (23306, 163) | Test  Y shape: (23306, 2)
Val   X shape: (16662, 163) | Val   Y shape: (16662, 2)


# Section 4

## SECTION 4.1 – Prepare data for XGBoost (all numeric) and CatBoost (with categoricals)

In [21]:
cat_cols = X_all.select_dtypes(include=["object", "category"]).columns
print("Categorical columns:")
print(list(cat_cols))

cat_indices = [X_all.columns.get_loc(c) for c in cat_cols]
print("\nIndices of categorical columns (for CatBoost):")
print(cat_indices)

X_all_xgb = pd.get_dummies(X_all, columns=cat_cols, drop_first=False)

print("\nShapes BEFORE and AFTER one-hot encoding:")
print("Original X_all shape: ", X_all.shape)
print("X_all_xgb shape:      ", X_all_xgb.shape)

X_train_xgb = X_all_xgb[train_mask]
X_test_xgb  = X_all_xgb[test_mask]
X_val_xgb   = X_all_xgb[val_mask]

X_train_cb = X_train.copy()
X_test_cb  = X_test.copy()
X_val_cb   = X_val.copy()

y_train_scores    = Y_train["scores"].astype(int).values
y_test_scores     = Y_test["scores"].astype(int).values
y_val_scores      = Y_val["scores"].astype(int).values

y_train_concedes  = Y_train["concedes"].astype(int).values
y_test_concedes   = Y_test["concedes"].astype(int).values
y_val_concedes    = Y_val["concedes"].astype(int).values

print("\nPrepared X and y for both models.")


Categorical columns:
['actiontype_a0', 'actiontype_a1', 'actiontype_a2', 'bodypart_a0', 'bodypart_a1', 'bodypart_a2', 'result_a0', 'result_a1', 'result_a2']

Indices of categorical columns (for CatBoost):
[0, 1, 2, 72, 73, 74, 87, 88, 89]

Shapes BEFORE and AFTER one-hot encoding:
Original X_all shape:  (128484, 163)
X_all_xgb shape:       (128484, 253)

Prepared X and y for both models.


## SECTION 4.2 – Helper function to evaluate probability models

In [22]:
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
import numpy as np

def evaluate_prob_model(y_true, y_prob, name="model"):
    eps = 1e-9
    y_prob = np.clip(y_prob, eps, 1 - eps)

    baseline = np.full_like(y_true, fill_value=y_true.mean(), dtype=float)

    brier = brier_score_loss(y_true, y_prob)
    brier_base = brier_score_loss(y_true, baseline)

    ll = log_loss(y_true, y_prob)
    ll_base = log_loss(y_true, baseline)

    auc = roc_auc_score(y_true, y_prob)

    print(f"=== {name} ===")
    print(f"Brier:   {brier:.4f}  (baseline {brier_base:.4f})")
    print(f"LogLoss: {ll:.4f}  (baseline {ll_base:.4f})")
    print(f"AUC:     {auc:.4f}")
    print()


## SECTION 4.3 – Train XGBoost and CatBoost for the 'scores' label

In [23]:
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

models_xgb = {}
models_cb  = {}

model_xgb_scores = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    random_state=RNG_SEED,
)
model_xgb_scores.fit(X_train_xgb, y_train_scores)
models_xgb["scores"] = model_xgb_scores


model_cb_scores = CatBoostClassifier(
    depth=6,
    learning_rate=0.05,
    iterations=500,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=RNG_SEED,
    verbose=False,
)
model_cb_scores.fit(
    X_train_cb,
    y_train_scores,
    cat_features=cat_indices,
)
models_cb["scores"] = model_cb_scores

p_train_xgb_scores = model_xgb_scores.predict_proba(X_train_xgb)[:, 1]
p_test_xgb_scores  = model_xgb_scores.predict_proba(X_test_xgb)[:, 1]

p_train_cb_scores = model_cb_scores.predict_proba(X_train_cb)[:, 1]
p_test_cb_scores  = model_cb_scores.predict_proba(X_test_cb)[:, 1]

p_train_ens_scores = 0.5 * p_train_xgb_scores + 0.5 * p_train_cb_scores
p_test_ens_scores  = 0.5 * p_test_xgb_scores  + 0.5 * p_test_cb_scores

print("SCORES – Train set performance:")
evaluate_prob_model(y_train_scores, p_train_xgb_scores, "XGBoost (scores, train)")
evaluate_prob_model(y_train_scores, p_train_cb_scores,  "CatBoost (scores, train)")
evaluate_prob_model(y_train_scores, p_train_ens_scores, "Ensemble (scores, train)")

print("SCORES – Test set performance:")
evaluate_prob_model(y_test_scores, p_test_xgb_scores, "XGBoost (scores, test)")
evaluate_prob_model(y_test_scores, p_test_cb_scores,  "CatBoost (scores, test)")
evaluate_prob_model(y_test_scores, p_test_ens_scores, "Ensemble (scores, test)")


SCORES – Train set performance:
=== XGBoost (scores, train) ===
Brier:   0.0040  (baseline 0.0104)
LogLoss: 0.0160  (baseline 0.0581)
AUC:     0.9988

=== CatBoost (scores, train) ===
Brier:   0.0068  (baseline 0.0104)
LogLoss: 0.0326  (baseline 0.0581)
AUC:     0.9316

=== Ensemble (scores, train) ===
Brier:   0.0052  (baseline 0.0104)
LogLoss: 0.0207  (baseline 0.0581)
AUC:     0.9971

SCORES – Test set performance:
=== XGBoost (scores, test) ===
Brier:   0.0108  (baseline 0.0125)
LogLoss: 0.0609  (baseline 0.0681)
AUC:     0.7365

=== CatBoost (scores, test) ===
Brier:   0.0107  (baseline 0.0125)
LogLoss: 0.0553  (baseline 0.0681)
AUC:     0.7741

=== Ensemble (scores, test) ===
Brier:   0.0107  (baseline 0.0125)
LogLoss: 0.0568  (baseline 0.0681)
AUC:     0.7623



## SECTION 4.4 – Train XGBoost and CatBoost for the 'concedes' label

In [24]:
model_xgb_concedes = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    random_state=RNG_SEED,
)
model_xgb_concedes.fit(X_train_xgb, y_train_concedes)
models_xgb["concedes"] = model_xgb_concedes

model_cb_concedes = CatBoostClassifier(
    depth=6,
    learning_rate=0.05,
    iterations=500,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=RNG_SEED,
    verbose=False,
)
model_cb_concedes.fit(
    X_train_cb,
    y_train_concedes,
    cat_features=cat_indices,
)
models_cb["concedes"] = model_cb_concedes

# Predictions on TRAIN and TEST
p_train_xgb_concedes = model_xgb_concedes.predict_proba(X_train_xgb)[:, 1]
p_test_xgb_concedes  = model_xgb_concedes.predict_proba(X_test_xgb)[:, 1]

p_train_cb_concedes = model_cb_concedes.predict_proba(X_train_cb)[:, 1]
p_test_cb_concedes  = model_cb_concedes.predict_proba(X_test_cb)[:, 1]

p_train_ens_concedes = 0.5 * p_train_xgb_concedes + 0.5 * p_train_cb_concedes
p_test_ens_concedes  = 0.5 * p_test_xgb_concedes  + 0.5 * p_test_cb_concedes

print("CONCEDES – Train set performance:")
evaluate_prob_model(y_train_concedes, p_train_xgb_concedes, "XGBoost (concedes, train)")
evaluate_prob_model(y_train_concedes, p_train_cb_concedes,  "CatBoost (concedes, train)")
evaluate_prob_model(y_train_concedes, p_train_ens_concedes, "Ensemble (concedes, train)")

print("CONCEDES – Test set performance:")
evaluate_prob_model(y_test_concedes, p_test_xgb_concedes, "XGBoost (concedes, test)")
evaluate_prob_model(y_test_concedes, p_test_cb_concedes,  "CatBoost (concedes, test)")
evaluate_prob_model(y_test_concedes, p_test_ens_concedes, "Ensemble (concedes, test)")


CONCEDES – Train set performance:
=== XGBoost (concedes, train) ===
Brier:   0.0002  (baseline 0.0026)
LogLoss: 0.0014  (baseline 0.0181)
AUC:     1.0000

=== CatBoost (concedes, train) ===
Brier:   0.0016  (baseline 0.0026)
LogLoss: 0.0092  (baseline 0.0181)
AUC:     0.9542

=== Ensemble (concedes, train) ===
Brier:   0.0007  (baseline 0.0026)
LogLoss: 0.0031  (baseline 0.0181)
AUC:     1.0000

CONCEDES – Test set performance:
=== XGBoost (concedes, test) ===
Brier:   0.0024  (baseline 0.0027)
LogLoss: 0.0170  (baseline 0.0184)
AUC:     0.7985

=== CatBoost (concedes, test) ===
Brier:   0.0024  (baseline 0.0027)
LogLoss: 0.0145  (baseline 0.0184)
AUC:     0.8042

=== Ensemble (concedes, test) ===
Brier:   0.0024  (baseline 0.0027)
LogLoss: 0.0149  (baseline 0.0184)
AUC:     0.8013



# Section 5

## SECTION 5.1 – Evaluate XGBoost, CatBoost, and Ensemble on the VALIDATION set

In [25]:
# SCORES
p_val_xgb_scores = model_xgb_scores.predict_proba(X_val_xgb)[:, 1]
p_val_cb_scores  = model_cb_scores.predict_proba(X_val_cb)[:, 1]
p_val_ens_scores = 0.5 * p_val_xgb_scores + 0.5 * p_val_cb_scores

print("SCORES – Validation set performance:")
evaluate_prob_model(y_val_scores, p_val_xgb_scores, "XGBoost (scores, val)")
evaluate_prob_model(y_val_scores, p_val_cb_scores,  "CatBoost (scores, val)")
evaluate_prob_model(y_val_scores, p_val_ens_scores, "Ensemble (scores, val)")

# CONCEDES
p_val_xgb_concedes = model_xgb_concedes.predict_proba(X_val_xgb)[:, 1]
p_val_cb_concedes  = model_cb_concedes.predict_proba(X_val_cb)[:, 1]
p_val_ens_concedes = 0.5 * p_val_xgb_concedes + 0.5 * p_val_cb_concedes

print("CONCEDES – Validation set performance:")
evaluate_prob_model(y_val_concedes, p_val_xgb_concedes, "XGBoost (concedes, val)")
evaluate_prob_model(y_val_concedes, p_val_cb_concedes,  "CatBoost (concedes, val)")
evaluate_prob_model(y_val_concedes, p_val_ens_concedes, "Ensemble (concedes, val)")


SCORES – Validation set performance:
=== XGBoost (scores, val) ===
Brier:   0.0061  (baseline 0.0074)
LogLoss: 0.0336  (baseline 0.0442)
AUC:     0.8384

=== CatBoost (scores, val) ===
Brier:   0.0059  (baseline 0.0074)
LogLoss: 0.0317  (baseline 0.0442)
AUC:     0.8727

=== Ensemble (scores, val) ===
Brier:   0.0060  (baseline 0.0074)
LogLoss: 0.0320  (baseline 0.0442)
AUC:     0.8660

CONCEDES – Validation set performance:
=== XGBoost (concedes, val) ===
Brier:   0.0030  (baseline 0.0034)
LogLoss: 0.0207  (baseline 0.0228)
AUC:     0.7697

=== CatBoost (concedes, val) ===
Brier:   0.0029  (baseline 0.0034)
LogLoss: 0.0187  (baseline 0.0228)
AUC:     0.7413

=== Ensemble (concedes, val) ===
Brier:   0.0029  (baseline 0.0034)
LogLoss: 0.0188  (baseline 0.0228)
AUC:     0.7688



## SECTION 5.2 – Compute final ensemble probabilities for ALL actions (train + test + val)

In [26]:
p_all_xgb_scores    = model_xgb_scores.predict_proba(X_all_xgb)[:, 1]
p_all_xgb_concedes  = model_xgb_concedes.predict_proba(X_all_xgb)[:, 1]

p_all_cb_scores     = model_cb_scores.predict_proba(X_all)[:, 1]
p_all_cb_concedes   = model_cb_concedes.predict_proba(X_all)[:, 1]

p_all_ens_scores    = 0.5 * p_all_xgb_scores   + 0.5 * p_all_cb_scores
p_all_ens_concedes  = 0.5 * p_all_xgb_concedes + 0.5 * p_all_cb_concedes

pred_all = pd.DataFrame({
    "game_id": game_ids.values,
    "scores": p_all_ens_scores,
    "concedes": p_all_ens_concedes,
})

pred_all.head()


,game_id,scores,concedes
0,7585,0.000564,0.000197
1,7585,0.000427,0.000261
2,7585,0.000416,0.000138
3,7585,0.000750,0.000180
4,7585,0.000895,0.000169


## SECTION 5.3 – Save ensemble predictions per game into predictions.h5

In [27]:
print("Saving predictions to:", PREDICTIONS_H5)

with pd.HDFStore(PREDICTIONS_H5, mode="w") as predstore:
    for gid, df_g in pred_all.groupby("game_id"):
        key = f"game_{int(gid)}"
      
        predstore[key] = df_g[["scores", "concedes"]]

print("Finished writing ensemble predictions to predictions.h5")


Saving predictions to: ./data-master\predictions.h5
Finished writing ensemble predictions to predictions.h5


# Section 6

## SECTION 6.0 – Ensure teams, players and player_games are stored in SPADL_H5

In [29]:
print("Rebuilding teams, players, and player_games from game-level API...")

teams_list = []
players_list = []

for game_id in tqdm(games.game_id, desc="Collecting team & player info"):
    try:
        teams_g = loader.teams(int(game_id))
        players_g = loader.players(int(game_id))

        teams_g["game_id"] = int(game_id)
        players_g["game_id"] = int(game_id)

        teams_list.append(teams_g)
        players_list.append(players_g)

    except Exception as e:
        print(f"Error loading game {game_id}: {e}")

teams_df   = pd.concat(teams_list, ignore_index=True).drop_duplicates()
players_df = pd.concat(players_list, ignore_index=True).drop_duplicates()

players_table = players_df[["player_id", "player_name", "nickname"]].drop_duplicates(subset="player_id")

player_games_table = players_df[
    ["player_id", "game_id", "team_id", "is_starter",
     "starting_position_id", "starting_position_name", "minutes_played"]
].drop_duplicates()

with pd.HDFStore(SPADL_H5) as store:
    store["teams"] = teams_df
    store["players"] = players_table
    store["player_games"] = player_games_table

print("Saved teams, players, and player_games into SPADL_H5 successfully!")
print("Teams:", teams_df.shape, "| Players:", players_table.shape, "| Player-games:", player_games_table.shape)

Rebuilding teams, players, and player_games from game-level API...


  warnings.warn(
C:\Users\Arya1\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsbombpy\api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
C:\Users\Arya1\AppData\Local\Programs\Python\Python311\Lib\site-packages\socceraction\data\statsbomb\loader.py:337: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  eventsdf["under_pressure"] = eventsdf["under_pressure"].fillna(False).astype(bool)
C:\Users\Arya1\AppData\Local\Programs\Python\Python311\Lib\site-packages\socceraction\data\statsbomb\loader.py:338: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.

Saved teams, players, and player_games into SPADL_H5 successfully!
Teams: (128, 3) | Players: (604, 3) | Player-games: (1790, 7)


## SECTION 6.1 – Compute VAEP per action using ensemble predictions

In [31]:
A_list = []

with pd.HDFStore(SPADL_H5) as spadlstore, pd.HDFStore(PREDICTIONS_H5) as predstore:
    for game in tqdm(list(games.itertuples()), desc="Computing VAEP values per action"):
        game_id = int(game.game_id)
        
        actions = spadlstore[f"actions/game_{game_id}"]
        actions_named = spadl.add_names(actions)

        preds = predstore[f"game_{game_id}"] 

        assert len(actions_named) == len(preds), f"Length mismatch in game {game_id}"

        actions_named = actions_named.reset_index(drop=True)
        preds = preds.reset_index(drop=True)

        values = vaepformula.value(
            actions_named,
            preds["scores"],
            preds["concedes"],
        )

        df = pd.concat(
            [
                actions_named,
                preds,
                values,
            ],
            axis=1,
        )
        df["game_id"] = game_id
        
        A_list.append(df)

A = pd.concat(A_list, ignore_index=True)

print("Full VAEP action table shape:", A.shape)
A.head()

Computing VAEP values per action: 100%|████████████████████████████████████████████████| 64/64 [00:01<00:00, 51.61it/s]

Full VAEP action table shape: (128484, 22)


,game_id,original_event_id,period_id,time_seconds,team_id,player_id,start_x,start_y,end_x,end_y,type_id,result_id,bodypart_id,action_id,type_name,result_name,bodypart_name,scores,concedes,offensive_value,defensive_value,vaep_value
0,7585,d4883f20-ce68-4f84-b26a-a049a13cb6be,1,0.24,769,3445.0,52.0625,34.425,43.3125,33.575,0,1,4,0,pass,success,foot_left,0.000564,0.000197,0.000000,-0.000000,0.000000
1,7585,b948f032-4c54-4782-a71a-ffeed8908d00,1,0.48,769,5692.0,43.3125,33.575,44.1875,34.425,21,1,0,1,dribble,success,foot,0.000427,0.000261,-0.000137,-0.000064,-0.000201
2,7585,9bdb71f9-c87b-4a66-96f0-def5312ca921,1,2.12,769,5692.0,44.1875,34.425,40.6875,22.525,0,1,4,2,pass,success,foot_left,0.000416,0.000138,-0.000011,0.000123,0.000112
3,7585,2ffa2904-8b47-4817-af26-aa9ac8d2881a,1,3.44,769,5685.0,40.6875,22.525,42.4375,21.675,21,1,0,3,dribble,success,foot,0.000750,0.000180,0.000334,-0.000042,0.000292
4,7585,6cb0d85d-bd14-42e3-9c2d-7f99ce437796,1,4.20,769,5685.0,42.4375,21.675,56.4375,1.275,0,1,5,4,pass,success,foot_right,0.000895,0.000169,0.000145,0.000010,0.000155


## Section 6.2 (player totals)

In [32]:
players      = pd.read_hdf(SPADL_H5, "players")
player_games = pd.read_hdf(SPADL_H5, "player_games")

A["action_count"] = 1

player_totals = (
    A[["player_id", "vaep_value", "offensive_value", "defensive_value", "action_count"]]
    .groupby("player_id")
    .sum()
    .reset_index()
)

player_names = players[["player_id", "player_name", "nickname"]].drop_duplicates(subset="player_id")
player_totals = player_totals.merge(player_names, on="player_id", how="left")

player_totals["name"] = player_totals["nickname"].fillna(player_totals["player_name"])

player_totals_sorted = player_totals.sort_values("vaep_value", ascending=False)
player_totals_sorted[["player_id", "name", "vaep_value", "offensive_value", "defensive_value", "action_count"]].head(15)


,player_id,name,vaep_value,offensive_value,defensive_value,action_count
50,3308.0,Kieran Trippier,4.056900,4.481715,-0.424815,684
6,3009.0,Kylian Mbappé,3.341797,3.315308,0.026489,489
92,3621.0,Eden Hazard,3.317112,3.347826,-0.030714,687
17,3089.0,Kevin De Bruyne,3.012714,3.012786,-0.000072,714
599,20004.0,Paul Pogba,2.456775,2.329828,0.126947,673
252,5474.0,Ivan Perišić,2.425029,2.561934,-0.136905,446
152,5186.0,Denis Cheryshev,2.355456,3.273142,-0.917686,213
71,3501.0,Philippe Coutinho,2.262762,2.182712,0.080050,696
250,5472.0,Mario Mandžukić,2.239054,2.840108,-0.601054,334
352,5574.0,Toni Kroos,2.186988,2.305046,-0.118058,641


## Section 6.3 (VAEP per 90 minutes)

In [33]:
minutes = (
    player_games[["player_id", "minutes_played"]]
    .groupby("player_id")
    .sum()
    .reset_index()
)

player_stats = player_totals.merge(minutes, on="player_id", how="left")
player_stats = player_stats[player_stats["minutes_played"] >= 180].copy()

player_stats["vaep_rating"]      = player_stats["vaep_value"] * 90 / player_stats["minutes_played"]
player_stats["offensive_rating"] = player_stats["offensive_value"] * 90 / player_stats["minutes_played"]
player_stats["defensive_rating"] = player_stats["defensive_value"] * 90 / player_stats["minutes_played"]

player_ratings = player_stats.sort_values("vaep_rating", ascending=False)

player_ratings[["player_id", "name", "minutes_played",
                "vaep_rating", "offensive_rating", "defensive_rating"]].head(15)


,player_id,name,minutes_played,vaep_rating,offensive_rating,defensive_rating
251,5473.0,Ahmed Musa,224,0.777921,0.784784,-0.006863
152,5186.0,Denis Cheryshev,317,0.668741,0.929283,-0.260542
352,5574.0,Toni Kroos,295,0.667217,0.703234,-0.036018
50,3308.0,Kieran Trippier,623,0.586069,0.647439,-0.061370
75,3531.0,Mohamed Salah,195,0.566807,0.579112,-0.012305
92,3621.0,Eden Hazard,551,0.541815,0.546832,-0.005017
6,3009.0,Kylian Mbappé,559,0.538035,0.533771,0.004265
121,4319.0,Edinson Cavani,362,0.536878,0.548598,-0.011720
525,6196.0,Yerry Mina,374,0.510317,0.426037,0.084280
16,3083.0,Son Heung-Min,294,0.506562,0.525101,-0.018539
